# Week 5 실습 — 결측값 처리와 범주형 변수의 가변수 처리

> "빈칸 많은 열이랑 이상한 값은 다 지우고 깔끔하게 정리해 주세요." — 팀장

이번 주 데이터는 **Ames, Iowa 주택 가격 데이터**(Kaggle House Prices)다. 이 데이터가 결측값과 가변수 처리에 좋은 이유가 세 가지 있다.

1. **구조적 결측:** `PoolQC`, `Alley`, `Fence`의 NA는 "수영장/골목/울타리가 없다"는 뜻이다. `isnull()`에는 잡히지만 진짜 결측이 아니다.
2. **진짜 결측:** `LotFrontage`(도로 접면 길이)는 정말 모르는 값이다. 같은 NA인데 의미가 다르다.
3. **순서형 범주:** `ExterQual`(외장재 품질)은 Ex > Gd > TA > Fa > Po 순서가 있다. 명목형(`Neighborhood`)과 처리 방식이 달라야 한다.

"NA는 다 같다"가 아니라 **왜 빈칸인지**에 따라 처리가 달라진다는 것을 확인한다.

교재: 11.1 결측값 처리, 11.7 범주형 변수의 가변수 처리

## 기본 세팅 [실행]

In [2]:
# 확인할 것: OK (1460, 81)이 출력되는지.
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100

# 저장소 루트의 data/ 폴더에서 찾기
_candidates = [Path("."), Path(".."), Path("../..")]
DATA_DIR = next((p.resolve() / "data" for p in _candidates if (p / "data").is_dir()), Path("."))

raw = pd.read_csv(DATA_DIR / "train.csv")
print("OK", raw.shape)

OK (1460, 81)


---
## Part 0. 도입 — 같은 NA, 다른 의미 [실행, 예측, 해석]


In [ ]:
# 확인할 것: 결측 비율 상위 5개 열. PoolQC가 99% 이상 결측인데, 정말 모르는 값인가?
miss = pd.DataFrame({
    "결측 수": raw.isnull().sum(),
    "결측 비율(%)": (raw.isnull().mean() * 100).round(1)
}).sort_values("결측 비율(%)", ascending=False)
display(miss.head(10))

In [ ]:
# 확인할 것: PoolQC가 NA인 집의 PoolArea(수영장 면적)가 거의 전부 0인지.
pool = raw[["PoolQC", "PoolArea"]].copy()
pool["PoolQC 결측"] = pool["PoolQC"].isnull()

display(pool.groupby("PoolQC 결측")["PoolArea"].describe().round(1))

print(f"\nPoolArea == 0인 집: {(raw['PoolArea'] == 0).sum()}건")
print(f"PoolQC가 NA인 집:   {raw['PoolQC'].isnull().sum()}건")

[예측] PoolQC의 NA 1,453건은 "수영장 품질을 모른다"인가, "수영장이 없다"인가? → (    )

[해석] PoolArea=0인 집과 PoolQC=NA인 집이 거의 일치한다. 이 NA를 `dropna()`로 지우면 무엇이 사라지는가?

-

---
## Part 1. 결측 현황 파악과 분류 [실행, 빈칸, 해석]

### 1-1. 결측 비율표 [실행]

In [ ]:
# 확인할 것: 결측 비율 상위 20개. 상위권이 대부분 "없음" 계열인지.
miss_top = miss[miss["결측 수"] > 0].head(20)

fig, ax = plt.subplots(figsize=(8, 7))
colors = []
# data_description.txt 기준: NA = "없음"인 변수
structural_na = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                 "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                 "BsmtExposure", "BsmtFinType2", "BsmtFinType1", "BsmtCond", "BsmtQual",
                 "MasVnrType"]
for col in miss_top.index:
    colors.append("steelblue" if col in structural_na else "coral")

bars = ax.barh(miss_top.index, miss_top["결측 비율(%)"], color=colors)
ax.set_xlabel("결측 비율 (%)")
ax.set_title("변수별 결측 비율 (파랑=구조적 결측, 빨강=진짜 결측)")
for i, v in enumerate(miss_top["결측 비율(%)"]):
    ax.text(v + 0.5, i, f"{v}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

### 1-2. 구조적 결측 확인 [빈칸]

교재 11.1에서 결측값은 완전 무작위 결측, 무작위 결측, 비무작위 결측 세 가지로 분류한다고 했다. 구조적 결측은 이 분류와 어떤 관계인가?

In [ ]:
# 확인할 것: 구조적 결측 변수들이 관련 수치 변수와 일치하는지 교차 확인.
checks = [
    ("GarageType", "GarageArea", "차고"),
    ("BsmtQual",   "TotalBsmtSF", "지하실"),
    ("FireplaceQu","Fireplaces",  "벽난로"),
    ("Alley",      None,          "골목"),
]

for qual_col, area_col, label in checks:
    na_count = raw[qual_col].isnull().sum()
    if area_col:
        zero_count = (raw[area_col] == 0).sum()
        match = (raw[qual_col].isnull() & (raw[area_col] == 0)).sum()
        print(f"{label}: {qual_col} NA={na_count}, {area_col}=0은 {zero_count}건, 겹침={match}건")
    else:
        print(f"{label}: {qual_col} NA={na_count}건")

[해석] 구조적 결측은 완전 무작위 결측, 무작위 결측, 비무작위 결측 중 어디에 해당하는가? 교재의 분류와 어떻게 다른가?

-

---
## Part 2. 구조적 결측 처리 [빈칸, 해석]

### 2-1. 구조적 결측 채우기 [빈칸]

In [ ]:
# 확인할 것: 구조적 결측을 "None"(범주형) 또는 0(수치형)으로 채운 뒤 결측이 줄어드는지.
df = raw.copy()

# 범주형 구조적 결측 → "None"
cat_structural = TODO  # data_description.txt에서 NA = "없음"인 범주형 변수를 리스트로 적는다
for col in cat_structural:
    df[col] = df[col].fillna("None")

# 수치형 구조적 결측 → 0
num_structural = ["MasVnrArea", "GarageYrBlt"]
for col in num_structural:
    df[col] = df[col].fillna(0)

print("===== 구조적 결측 처리 후 =====")
remaining = df.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(f"결측 있는 열: {len(remaining)}개")
display(remaining)

[해석] 구조적 결측을 처리하니 결측 있는 열이 몇 개로 줄었는가? 남은 결측은 어떤 변수인가?

-

---
## Part 3. 진짜 결측 처리 비교 [예측, 빈칸, 해석]

### 3-0. 예측 [예측]

`LotFrontage`(도로 접면 길이, 결측 약 17%)에 전체 평균으로 대치하면 분포는 어떻게 바뀔까?
- 변하지 않는다 / 평균 부근에 봉우리가 생긴다 / 전체적으로 균일해진다 → (    )

In [ ]:
# 확인할 것: 제거 vs 평균 대치 vs 중앙값 대치 후 분포와 표준편차 비교.
col = "LotFrontage"
original = df[col].dropna()
df_mean = df[col].fillna(df[col].mean())
df_median = df[col].fillna(df[col].median())

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, title, data in zip(axes,
    ["원본 (결측 제외)", "평균 대치", "중앙값 대치"],
    [original, df_mean, df_median]):
    ax.hist(data, bins=40, edgecolor="white", alpha=0.7)
    ax.axvline(data.mean(), color="red", linestyle="--", label=f"평균 {data.mean():.1f}")
    ax.axvline(data.median(), color="blue", linestyle="--", label=f"중앙값 {data.median():.1f}")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlabel("LotFrontage (ft)")
axes[0].set_ylabel("빈도")
plt.suptitle("LotFrontage 결측값 대치 방법 비교", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"원본 std:     {original.std():.2f}")
print(f"평균 대치 std: {df_mean.std():.2f}  (변화율: {(df_mean.std()-original.std())/original.std()*100:.1f}%)")
print(f"중앙값 대치 std: {df_median.std():.2f}  (변화율: {(df_median.std()-original.std())/original.std()*100:.1f}%)")

[해석] 표준편차가 줄어든 이유는? 이것이 분석에 어떤 영향을 주는가?

-

### 3-2. 그룹별 대치 — Neighborhood별 중앙값 [빈칸]

In [ ]:
# 확인할 것: 동네마다 LotFrontage 중앙값이 다른지, 그룹별 대치가 전체 대치보다 원본에 가까운지.
print("Neighborhood별 LotFrontage 중앙값 (상위 10):")
nb_median = df.groupby("Neighborhood")[col].median().sort_values(ascending=False)
display(nb_median.head(10))

# 그룹별 중앙값 대치
df_group = df.copy()
df_group[col] = TODO  # Neighborhood별 중앙값으로 LotFrontage 결측을 채운다 (groupby + transform + fillna)
# 그룹 내에서도 전부 결측이면 전체 중앙값으로
df_group[col] = df_group[col].fillna(df[col].median())

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, title, data in zip(axes,
    ["원본 (결측 제외)", "전체 중앙값 대치", "동네별 중앙값 대치"],
    [original, df_median, df_group[col]]):
    ax.hist(data, bins=40, edgecolor="white", alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("LotFrontage (ft)")
axes[0].set_ylabel("빈도")
plt.suptitle("전체 vs 그룹별 대치 비교", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n원본 std:         {original.std():.2f}")
print(f"전체 중앙값 std:   {df_median.std():.2f}")
print(f"동네별 중앙값 std: {df_group[col].std():.2f}")

[해석] 그룹별 대치의 표준편차가 전체 대치보다 원본에 가까운가? 왜 그런가?

-

---
## Part 4. 보간법과 다중 대치 [실행, 빈칸, 해석]

### 4-1. 보간법 — 시계열 데이터 예시 [실행]

보간법은 시계열 특성이 있을 때 효과적이다. 교재 그림 11.2와 같은 예시로 확인한다.

In [ ]:
# 확인할 것: 단순 순서 보간과 시점 고려 보간의 대치 값이 다른지.
dates = pd.date_range("2022-04-17", periods=8, freq="D")
sales = pd.Series([2800, 3100, 3000, np.nan, np.nan, np.nan, 3900, 4200], index=dates)

interp = pd.DataFrame({
    "원본": sales,
    "선형 보간": sales.interpolate(method="linear"),
    "시점 보간": sales.interpolate(method="time")
})
display(interp)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dates, sales, "ko-", label="원본", ms=8)
ax.plot(dates, interp["선형 보간"], "b^--", label="선형 보간", ms=7)
ax.plot(dates, interp["시점 보간"], "rs--", label="시점 보간", ms=7)
for d in dates[sales.isnull()]:
    ax.axvspan(d - pd.Timedelta("6h"), d + pd.Timedelta("6h"), alpha=0.15, color="gray")
ax.legend()
ax.set_title("보간법 비교 — 매출 데이터 (교재 그림 11.2)")
ax.set_ylabel("매출(만)")
plt.tight_layout()
plt.show()

[해석] 주택 데이터에 보간법을 쓸 수 있는가? 어떤 조건일 때 보간법이 적절한가?

-

### 4-2. 다중 대치 [빈칸]

In [ ]:
# 확인할 것: IterativeImputer 적용 후 LotFrontage 분포가 평균 대치보다 원본에 가까운지.
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

num_cols = ["LotFrontage", "LotArea", "OverallQual", "OverallCond",
            "YearBuilt", "TotalBsmtSF", "GrLivArea", "SalePrice"]
df_num = df[num_cols].copy()

imputer = IterativeImputer(max_iter=10, random_state=42, n_nearest_features=5)
df_mice = pd.DataFrame(imputer.fit_transform(df_num), columns=num_cols)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
axes[0].hist(df[col].dropna(), bins=40, alpha=0.7, edgecolor="white")
axes[0].set_title("원본 (결측 제외)")
axes[1].hist(df[col].fillna(df[col].mean()), bins=40, alpha=0.7, color="orange", edgecolor="white")
axes[1].set_title("평균 대치")
axes[2].hist(df_mice["LotFrontage"], bins=40, alpha=0.7, color="green", edgecolor="white")
axes[2].set_title("다중 대치")
for ax in axes:
    ax.set_xlabel("LotFrontage (ft)")
axes[0].set_ylabel("빈도")
plt.suptitle("대치 방법별 LotFrontage 분포 비교", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"원본 std:     {df[col].dropna().std():.2f}")
print(f"평균 대치 std: {df[col].fillna(df[col].mean()).std():.2f}")
print(f"다중 대치 std: {df_mice['LotFrontage'].std():.2f}")

[해석] 다중 대치가 평균 대치보다 분포를 잘 보존하는가? 표준편차를 비교한다.

-

---
## Part 5. 범주형 변수의 가변수 처리 [실행, 빈칸, 해석]

### 5-1. 범주별 분포 확인 [실행]

In [ ]:
# 확인할 것: 각 범주형 변수의 범주 수. Neighborhood처럼 범주가 많은 변수가 있는지.
cat_examples = ["MSZoning", "Neighborhood", "BldgType", "ExterQual", "Foundation", "CentralAir"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, c in enumerate(cat_examples):
    counts = df[c].value_counts()
    axes[i].barh(counts.index.astype(str), counts.values)
    axes[i].set_title(f"{c} ({len(counts)}개 범주)")
    axes[i].tick_params(labelsize=8)
plt.suptitle("범주형 변수의 범주별 분포", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 5-2. 명목형 — get_dummies [빈칸]

In [ ]:
# 확인할 것: 전체 일괄 처리 vs 특정 열 지정 시 열 수 차이.
df_cat = df[["MSZoning", "BldgType", "CentralAir", "Foundation", "SalePrice"]].copy()

print(f"원본 열 수: {df_cat.shape[1]}")

# 전체 일괄 처리
df_d1 = pd.get_dummies(df_cat)
print(f"전체 가변수 처리 후: {df_d1.shape[1]}열")

# 특정 열만 지정
df_d2 = pd.get_dummies(df_cat, columns=["MSZoning", "CentralAir"])
print(f"특정 열만 처리 후: {df_d2.shape[1]}열")
df_d2.head()

### 5-3. CentralAir는 2개 범주 — 가변수가 몇 개 필요한가? [해석]

`CentralAir`(중앙 에어컨 유무)는 Y/N 두 범주다. 가변수 2개(`CentralAir_N`, `CentralAir_Y`)가 다 필요한가?

-

### 5-4. 순서형 — 숫자 매핑 [빈칸]

`ExterQual`(외장재 품질)은 Ex > Gd > TA > Fa > Po 순서가 있다. 교재 11.7에서 "선형 회귀분석이나 로지스틱 회귀분석 등의 회귀분석은 기본적으로 연속형 변수만 사용할 수 있다"고 했는데, 순서형은 원-핫보다 **숫자 매핑**이 정보를 더 보존한다.

In [ ]:
# 확인할 것: 순서형 변수를 숫자로 매핑한 후 SalePrice와의 관계가 단조 증가/감소인지.
qual_map = TODO  # Po, Fa, TA, Gd, Ex를 1~5로 매핑하는 딕셔너리를 만든다
ordinal_cols = ["ExterQual", "ExterCond", "BsmtQual", "KitchenQual", "HeatingQC"]

df_ord = df.copy()
for c in ordinal_cols:
    if c in df_ord.columns:
        df_ord[c + "_num"] = df_ord[c].map(qual_map)

# ExterQual 숫자 매핑 vs SalePrice
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

order = ["Po", "Fa", "TA", "Gd", "Ex"]
existing = [v for v in order if v in df["ExterQual"].values]
sns.boxplot(data=df, x="ExterQual", y="SalePrice", order=existing, ax=axes[0])
axes[0].set_title("ExterQual vs SalePrice (순서형)")
axes[0].set_xlabel("외장재 품질")

corrs = []
for c in ordinal_cols:
    num_col = c + "_num"
    if num_col in df_ord.columns:
        r = df_ord[[num_col, "SalePrice"]].dropna().corr().iloc[0,1]
        corrs.append({"변수": c, "SalePrice 상관계수": round(r, 3)})
corr_df = pd.DataFrame(corrs).sort_values("SalePrice 상관계수", ascending=False)
display(corr_df)

axes[1].barh(corr_df["변수"], corr_df["SalePrice 상관계수"])
axes[1].set_title("순서형 변수의 SalePrice 상관계수")
axes[1].set_xlabel("피어슨 상관계수")
plt.tight_layout()
plt.show()

[해석] 순서형 변수를 원-핫 인코딩하면 이 순서 정보를 잃는다. 어떤 경우에 순서형 매핑이 더 적절한가?

-

---
## Part 6. drop_first — 다중공선성 방지 [빈칸, 해석]


In [ ]:
# 확인할 것: drop_first 전후 열 수 차이, 빠진 baseline 범주.
nominal_cols = ["MSZoning", "BldgType", "CentralAir", "Foundation"]
df_cat2 = df[nominal_cols + ["SalePrice"]].copy()

df_no_drop = pd.get_dummies(df_cat2, columns=nominal_cols)
df_drop = pd.get_dummies(df_cat2, columns=nominal_cols, TODO)  # 교재 코드 11.64의 옵션을 적용한다

print(f"drop_first=False: {df_no_drop.shape[1]}열")
print(f"drop_first=True:  {df_drop.shape[1]}열  (줄어든 열: {df_no_drop.shape[1] - df_drop.shape[1]}개)")

for c in nominal_cols:
    cats = sorted(df_cat2[c].dropna().unique().astype(str))
    print(f"  {c}: {len(cats)}개 범주 → {len(cats)-1}개 가변수 (baseline: {cats[0]})")

[해석] 교재 그림 11.16의 혈액형 예시처럼, baseline 범주의 종속변수에 대한 영향력은 0으로 맞춰진다. CentralAir에서 baseline이 'N'이면 CentralAir_Y 계수는 어떻게 해석하는가?

-

---
## Part 7. 전처리 파이프라인 [빈칸, 해석]


In [ ]:
# 확인할 것: 전처리 전후 결측 수와 열 수 변화.
df_pipe = raw.copy()

# TODO: 아래 4단계를 직접 채운다. Part 2~6에서 한 것을 순서대로 적용한다.
# 1단계: 구조적 결측 처리 (범주형 → "None", 수치형 → 0)
# 2단계: 진짜 결측 대치 (LotFrontage는 동네별 중앙값, 나머지 범주형은 최빈값)
# 3단계: 순서형 → 숫자 매핑
# 4단계: 명목형 → 가변수 (drop_first=True)


print("===== 전처리 결과 =====")
print(f"  행: {df_pipe.shape[0]:,}")
print(f"  열: {df_pipe.shape[1]}")
print(f"  결측: {df_pipe.isnull().sum().sum()}건")

[해석] 전처리 순서가 중요한 이유는? 순서를 바꾸면 어떤 문제가 생기는가?

-

---
## Part 8. AI와 비교 [빈칸, 해석]

내 분석을 끝낸 뒤, AI에게 아래 프롬프트를 그대로 넣고 결과를 점검한다.

> "이 데이터에서 결측값을 처리하고 범주형 변수를 인코딩해줘."

| 점검 항목 | 통과 / 실패 | 근거 (AI 출력에서 인용) |
|---|---|---|
| 구조적 결측("없음")과 진짜 결측("모름")을 구분하는가 |  |  |
| PoolQC 99% 결측을 그냥 지우는가, "None"으로 채우는가 |  |  |
| 순서형(ExterQual)과 명목형(MSZoning)을 다르게 처리하는가 |  |  |
| drop_first를 적용하는가 |  |  |
| 대치 전후 분포 변화를 확인하는가 |  |  |

### 토론 [해석]

① 팀장이 "빈칸 많은 열 다 지워"라고 했을 때, PoolQC(99% NA)와 LotFrontage(17% NA)를 같은 기준으로 지워도 되는가?

-

② 순서형 변수의 간격이 등간(TA→Gd와 Gd→Ex의 차이가 같은지)인지 모르는데 1\~5로 매핑해도 되는가?

-

③ Neighborhood가 25개 범주인데 drop_first로 24개 가변수를 만들면 열이 너무 많아진다. 줄이는 방법은?

-

---
## 마무리 [해석]

오늘의 결론 세 줄. 결론을 바꾼 세 가지 선택을 오늘 본 숫자로 적는다.

1. 구조적 결측을 "없음"으로 보느냐 "결측"으로 보느냐 →
2. 무엇으로 채우느냐 (평균 / 중앙값 / 그룹별 / 다중 대치) →
3. 순서형을 숫자로 매핑하느냐 원-핫하느냐, drop_first를 쓰느냐 안 쓰느냐 →

PR 전 체크리스트:
- [ ] 구조적 결측과 진짜 결측을 구분했는가 (근거: data_description.txt 또는 관련 수치 변수)
- [ ] 대치 전후 분포 비교에서 표준편차를 언급했는가
- [ ] 순서형 변수의 매핑 근거를 적었는가
- [ ] drop_first의 baseline 범주를 명시했는가
- [ ] AI 비교 표를 채웠는가